# 01_data_generation_notebook

# Data Generation – Simulated IoT Sensor Ingestion

This notebook generates synthetic IoT sensor data with dirty records
and saves it as a CSV file in DBFS.


In [ ]:
from datetime import datetime, timedelta
import random


In [ ]:
sensor_data = []
base_time = datetime.now()

for i in range(60):
    sensor_data.append({
        "sensor_id": f"SENSOR_{random.randint(1,5)}",
        "event_ts": (base_time - timedelta(minutes=i*10)).strftime("%Y-%m-%d %H:%M:%S"),
        "temperature": random.choice([random.uniform(60, 95), None]),
        "humidity": random.uniform(30, 90),
        "voltage": random.choice([random.uniform(2.5, 5.0), -1.0])
    })

# Add dirty rows
sensor_data.append({
    "sensor_id": None,
    "event_ts": base_time.strftime("%Y-%m-%d %H:%M:%S"),
    "temperature": 85.0,
    "humidity": 50.0,
    "voltage": 3.3
})

sensor_data.append(sensor_data[0])  # duplicate record


In [ ]:
# Create managed volume in workspace catalog's default schema
spark.sql(
    "CREATE VOLUME IF NOT EXISTS workspace.default.raw"
)

csv_path = "/Volumes/workspace/default/raw/raw_sensor_data.csv"

csv_content = "sensor_id,event_ts,temperature,humidity,voltage\n"
for row in sensor_data:
    csv_content += (
        f"{row['sensor_id']},{row['event_ts']},{row['temperature']},{row['humidity']},{row['voltage']}\n"
    )

dbutils.fs.put(
    csv_path,
    csv_content,
    overwrite=True
)

display(f"Raw sensor data saved to: {csv_path}")